# バッチではない場合

In [ ]:
import numpy as np

u = np.array([[0.5],
              [0.5]])
v = np.array([[0.5],
              [0.5]])
C = np.array([[0.0, 1.0],
              [1.0, 0.0]])
T = u @ v.T

beta = 0.1
K = 100
S = 100
for k in range(K):
    loss = np.sum( T * C )

    G = T * np.exp(-1 * C / beta)
    a = np.ones(shape = u.shape)
    b = np.ones(shape = v.shape)

    for s in range(S):
        a = u / (G @ b)
        b = v / (G.T @ a)

    T = np.diag(a[:, 0]) @ G @ np.diag(b[:, 0])

T

# バッチの場合

In [36]:
import numpy as np

C = np.array([
    [[0.0, 1.0],
     [1.0, 0.0]],
    [[1.0, 0.0],
     [0.0, 1.0]]
])
B, S, R = C.shape

u = np.ones(shape = (B, S)) / S
v = np.ones(shape = (B, R)) / R

T = u[..., np.newaxis] @ np.transpose(v[..., np.newaxis], (0, 2, 1))

LAMBDA = 0.1

for _ in range(100):
    a = np.ones(shape = (B, S))
    b = np.ones(shape = (B, R))

    G = T * np.exp(-1.0 * C / LAMBDA)

    for _ in range(100):
        a = u / ( G @ b[..., np.newaxis] )[:, :, 0]
        b = v / ( np.transpose(G, (0, 2, 1)) @ a[..., np.newaxis] )[:, :, 0]

    # a = np.array([np.diag(A) for A in a])
    # b = np.array([np.diag(B) for B in b])
    # T = a @ G @ b

    T = a[..., np.newaxis] * G * b[:, np.newaxis, :]

T

array([[[0.5, 0. ],
        [0. , 0.5]],

       [[0. , 0.5],
        [0.5, 0. ]]])

# PyTorchの場合

In [37]:
import torch

C = torch.tensor([
    [[0.0, 1.0],
     [1.0, 0.0]],
    [[1.0, 0.0],
     [0.0, 1.0]]
])
B, S, R = C.shape

u = torch.ones(size = (B, S)) / S
v = torch.ones(size = (B, R)) / R

T = u.unsqueeze(-1) @ v.unsqueeze(-1).permute(0, 2, 1)

LAMBDA = 0.1

for _ in range(100):
    a = torch.ones(size = (B, S))
    b = torch.ones(size = (B, R))

    G = T * torch.exp(-1.0 * C / LAMBDA)

    for _ in range(100):
        a = u / ( G @ b.unsqueeze(-1) ).squeeze(-1)
        b = v / ( G.permute(0, 2, 1) @ a.unsqueeze(-1) ).squeeze(-1)

    T = a.unsqueeze(-1) * G * b.unsqueeze(1)

T

tensor([[[0.5000, 0.0000],
         [0.0000, 0.5000]],

        [[0.0000, 0.5000],
         [0.5000, 0.0000]]])

# PyTorchの関数

In [38]:
A = torch.rand(size = (2, 4, 5))
B = torch.rand(size = (2, 6, 5))

C = A @ B.permute(0, 2, 1)

def IPOT(C, LAMBDA=1.0, iters=50, inner_iters=10, eps=1e-8):
    B, S, R = C.shape

    u = torch.ones(size = (B, S)) / S
    v = torch.ones(size = (B, R)) / R

    T = u.unsqueeze(-1) @ v.unsqueeze(-1).permute(0, 2, 1)
    T.shape

    for _ in range(100):
        a = torch.ones(size = (B, S))
        b = torch.ones(size = (B, R))

        G = T * torch.exp(-1.0 * C / LAMBDA)

        for _ in range(100):
            a = u / ( G @ b.unsqueeze(-1) ).squeeze(-1)
            b = v / ( G.permute(0, 2, 1) @ a.unsqueeze(-1) ).squeeze(-1)

        T = a.unsqueeze(-1) * G * b.unsqueeze(1)

    return T

T = IPOT(C)
T.shape, T.sum(dim = -1).sum(dim = -1)

(torch.Size([2, 4, 6]), tensor([1., 1.]))